# Imagens coloridas e espaços de cor

Este notebook explora a representação BGR/RGB, a separação de canais, histogramas por canal e conversões entre espaços de cor.

## Objetivos

- reconhecer a diferença entre BGR (OpenCV) e RGB (Matplotlib);
- separar e recombinar os canais preservando sua ordem;
- converter imagens entre RGB, BGR, HSV, YCrCb, Lab e grayscale;
- calcular e visualizar histogramas por canal;
- aplicar uma transformação de intensidade independentemente em cada canal.

## 1. Instalação

A instalação usa a versão commitada na branch principal do repositório.

In [ ]:
%pip install -q "git+https://github.com/tfvieira/dip-2026-2.git"

In [ ]:
import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np

from dip_toolkit import download_course_image
from dip_toolkit.modules.image_color_processor import ColorImageProcessor
from dip_toolkit.modules.image_loader import ImageLoader

processor = ColorImageProcessor()
loader = ImageLoader()

## 2. BGR para processamento, RGB para exibição

O `ImageLoader` usa a convenção BGR do OpenCV para imagens coloridas. O Matplotlib, por sua vez, espera RGB. Por isso, a conversão é explícita antes de chamar `imshow`.

In [ ]:
image_path = download_course_image("astronaut.png")
image_bgr = loader.load_image(image_path, flags=cv.IMREAD_COLOR)
image_rgb = processor.convert(image_bgr, "bgr", "rgb")

print(f"BGR: shape={image_bgr.shape}, dtype={image_bgr.dtype}")
print(f"RGB: shape={image_rgb.shape}, dtype={image_rgb.dtype}")

plt.figure(figsize=(5, 5))
plt.imshow(image_rgb)
plt.title("Imagem exibida em RGB")
plt.axis("off")
plt.show()

## 3. Separação e recombinação de canais

`split_channels` retorna os três arrays 2D e a ordem dos canais. Esse metadado é usado por `combine_channels`, evitando a recombinação acidental em uma ordem diferente.

In [ ]:
channels = processor.split_channels(image_bgr, "bgr")
image_recombined = processor.combine_channels(channels)

print("Ordem preservada:", channels.channel_order)
print("Recombinação idêntica à entrada:", np.array_equal(image_bgr, image_recombined))

figure, axes = plt.subplots(1, 3, figsize=(13, 4))
for axis, channel, title in zip(axes, channels.channels, ("B", "G", "R"), strict=True):
    axis.imshow(channel, cmap="gray", vmin=0, vmax=255)
    axis.set_title(f"Canal {title}")
    axis.axis("off")
figure.tight_layout()
plt.show()

## 4. Espaços de cor

HSV usa H em `[0, 179]` e S/V em `[0, 255]`; YCrCb e Lab usam três canais `uint8` em `[0, 255]`. Os valores desses espaços não devem ser exibidos diretamente como RGB. Aqui, eles são usados como representação numérica; a imagem grayscale pode ser exibida com `cmap="gray"`.

In [ ]:
image_hsv = processor.convert(image_bgr, "bgr", "hsv")
image_ycrcb = processor.convert(image_bgr, "bgr", "ycrcb")
image_lab = processor.convert(image_bgr, "bgr", "lab")
image_gray = processor.convert(image_bgr, "bgr", "gray")

color_spaces = {
    "HSV": image_hsv,
    "YCrCb": image_ycrcb,
    "Lab": image_lab,
    "Gray": image_gray,
}
for name, image in color_spaces.items():
    description = f"{name}: shape={image.shape}, dtype={image.dtype}"
    interval = f"intervalo=[{image.min()}, {image.max()}]"
    print(f"{description}, {interval}")

plt.figure(figsize=(5, 5))
plt.imshow(image_gray, cmap="gray", vmin=0, vmax=255)
plt.title("Conversão para grayscale")
plt.axis("off")
plt.show()

## 5. Histogramas por canal

Cada histograma possui 256 posições. Para uma imagem BGR, as chaves retornadas são `b`, `g` e `r`, nessa ordem, e a soma de cada histograma é o número de pixels do canal correspondente.

In [ ]:
histograms = processor.channel_histograms(image_bgr, "bgr")
for channel, histogram in histograms.items():
    print(f"{channel.upper()}: {histogram.sum()} pixels")

figure, _ = processor.plot_channel_histograms(histograms, "bgr")
figure.tight_layout()
plt.show()

## 6. Operação por canal

As transformações de intensidade reutilizam os contratos de `IntensityTransformer`, mas são aplicadas separadamente em cada canal. A operação negativa abaixo preserva shape, dtype e a ordem BGR.

In [ ]:
negative_bgr = processor.apply_channel_operation(image_bgr, "bgr", "negative")
negative_rgb = processor.convert(negative_bgr, "bgr", "rgb")

figure, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(image_rgb)
axes[0].set_title("Original")
axes[1].imshow(negative_rgb)
axes[1].set_title("Negativa por canal")
for axis in axes:
    axis.axis("off")
figure.tight_layout()
plt.show()

## Exercício final

1. Carregue outra imagem colorida do acervo da disciplina.
2. Compare os histogramas BGR e RGB da mesma imagem.
3. Aplique uma transformação gamma por canal e descreva o efeito visual.
4. Explique por que HSV, YCrCb e Lab não devem ser desenhados diretamente com `imshow` como se fossem RGB.